In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import json

from dotenv import load_dotenv
load_dotenv()

import ollama

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import InjectedState

from file_tools import (
    DeepAgentState,
    ls,
    read_file,
    write_file,
    cleanup_files,
)
from prompts import (
    ORCHESTRATOR_PROMPT,
    RESEARCHER_PROMPT,
    EDITOR_PROMPT,
)

In [3]:
# -------------------------
# Web Search Tool
# -------------------------

@tool
def web_search(query: str) -> str:
    """
    Perform a live web search using Ollama Cloud Web Search API.

    Input:
        query: search query string

    Output:
        JSON string of top results (max_results=2).
    """
    response = ollama.web_search(query, max_results=2)
    response = response.results

    return response


In [4]:
# result = web_search.invoke({"query": "Latest global news"})

In [5]:
# -------------------------
# LLM
# -------------------------

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

In [6]:
# -------------------------
# Worker Agents: Researcher & Editor
# -------------------------

researcher_tools = [ls, write_file, read_file, web_search]
researcher_agent = create_agent(
    model=llm,
    tools=researcher_tools,
    system_prompt=RESEARCHER_PROMPT,
    state_schema=DeepAgentState,
)

editor_tools = [ls, read_file, write_file, cleanup_files]
editor_agent = create_agent(
    model=llm,
    tools=editor_tools,
    system_prompt=EDITOR_PROMPT,
    state_schema=DeepAgentState,
)


In [7]:
# -------------------------
# Orchestrator Tools (call other agents)
# -------------------------
from typing import Annotated
from langgraph.prebuilt import InjectedState

@tool
def run_researcher(
    state: Annotated[DeepAgentState, InjectedState]
) -> str:
    """
    Run the Research agent for this user/thread using the current conversation.

    The Researcher will:
    - use web_search internally
    - write plan.md, sources.txt, outline.md

    Returns a short status string for the orchestrator.
    """
    sub_state: DeepAgentState = {
        "messages": state["messages"],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    researcher_agent.invoke(sub_state)
    return "Research completed. Files plan.md, sources.txt, and outline.md are updated."



In [8]:
@tool
def run_editor(
    state: Annotated[DeepAgentState, InjectedState]
) -> str:
    """
    Run the Editor agent for this user/thread.

    The Editor will:
    - read plan.md, sources.txt, outline.md
    - write the final answer to report.md

    Returns a short status string for the orchestrator.
    """
    sub_state: DeepAgentState = {
        "messages": state["messages"],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    editor_agent.invoke(sub_state)
    return "Editor completed. Final report is written to report.md."


In [9]:
# -------------------------
# Orchestrator Agent
# -------------------------

orchestrator_tools = [
    web_search,      # for light research
    run_researcher,  # for deep research
    run_editor,      # for deep research
    ls,
    read_file,
    cleanup_files,
]

orchestrator_agent = create_agent(
    model=llm,
    tools=orchestrator_tools,
    system_prompt=ORCHESTRATOR_PROMPT,
    state_schema=DeepAgentState,
)



### Example usage


In [10]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Tell me a joke about computers.")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)
result

c:\Users\laxmi\anaconda3\envs\ml\Lib\site-packages\pydantic\v1\main.py:1054: UserWarning: LangSmith now uses UUID v7 for run and trace identifiers. This warning appears when passing custom IDs. Please use: from langsmith import uuid7
            id = uuid7()
Future versions will require UUID v7.
  input_data = validator(cls_, input_data)


{'messages': [HumanMessage(content='Tell me a joke about computers.', additional_kwargs={}, response_metadata={}, id='141c5283-ce9b-4a97-8937-1a8e1b09ee7e'),
  AIMessage(content='Why was the computer cold? Because it left its Windows open!', additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--f733a8f8-49c4-4b3b-b409-2e17abe495f6-0', usage_metadata={'input_tokens': 1429, 'output_tokens': 13, 'total_tokens': 1442, 'input_token_details': {'cache_read': 0}})],
 'user_id': 'user_123',
 'thread_id': 'thread_mcp_001'}

In [11]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Do a detailed, well-structured analysis of MCP including history")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)

In [13]:
result

{'messages': [HumanMessage(content='Do a detailed, well-structured analysis of MCP including history', additional_kwargs={}, response_metadata={}, id='71f7fafc-8f91-4c3c-93a8-5c99f0623c4d'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'run_researcher', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'4cc60f01-aad6-4543-b25f-f4e9cd2a5802': 'CpYCAXLI2nyXInNVjQwuVoCsUAnhl/QhFC55EE4QfSleuzszt25GuI/D8PCqhEOQlzvJ5Uf8qzlZGDEXGHhijZJnU+G/N5E/zrK2PMbTYRLCh5uGbpecizzt9n+IsroLGUTi4h/ryTGk2+ZCZeoQrjR3HbNnJ5Mcm4nMbM4HvMs0KmsLCagxpn08bCLirFmgMQLEF3tDle+VQLKi1Ilsaky/VTr+PErQ74oitIvTAmm1MQkHC1lTkTXkYmbtj2UMxUan+THs3Me3Q5cyUN8ueBrXIETXRl2T8DzATmL7xzgVeHCqeruZHmxxU6Yg1264Qd/6s0rlp1Y7o9+NBxJVv7tOUFLuF1pPQU+nSkewHLnCjIB+8BIapa4='}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--42f68302-b201-4af4-a374

In [14]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Do a detailed, well-structured comparison of MCP and API.")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)

In [15]:
result

{'messages': [HumanMessage(content='Do a detailed, well-structured comparison of MCP and API.', additional_kwargs={}, response_metadata={}, id='2bf3423f-1b92-4bc8-8851-2bf69ca9acbb'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'run_researcher', 'arguments': '{}'}, '__gemini_function_call_thought_signatures__': {'dcdbee9e-a49d-4b5d-be31-1b65b2b6b5a1': 'CrICAXLI2nzbLY61IfeX7koORg7RiM2yXn6rlF5CdWoA6iRjnnjRadIyPDcnrWdjZUAdNQzjJHfjDdRSgCuj9wGvsYRYhyuP5M+k0syK8m/QfrVMlM5rA/iJmop0zIjAx5vdMvBoVoPozkfml5gpggYY5zjO2gSk1yHqwxXEQx7bjidnSSe9uIjAP1ny9zmDjUfvxONnKDYysu3iL6fRBWbSWugY+CIT/5L/IInuJ+sbVDw4tyHLJQAagVz0OYhLGxWsNffyePkx+K2Z2iXaTHNcBiDiIoJGqD3b/e1D7ELG/JNYF/ouZZy/N2MsJdZ8xTA9jetL/qcrkuBAw6YSS5OW7m+/wDzVmAE6+vBIdaEXC9CwuHhLcE7c91+g7JHOIV2+cC8/LOV83MjJl/9zOrRflTnp'}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc